[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/18.%20The%20Yard%20Crane%20%28RTG_RMG%29%20Scheduling%20Problem/P18-Tier-7_executed.ipynb)

---


# 18. The Yard Crane (RTG/RMG) Scheduling Problem
## Tier 7: Human–AI Partnership

### Purpose and scope

This notebook illustrates a **human–AI decision partnership** in a yard-crane operations context. It simulates how decision authority, recommendation quality, trust, and performance metrics evolve across a sequence of operational decisions.

The notebook does **not** formulate or solve a yard-crane scheduling optimization problem: it does not assign cranes to jobs, sequence moves, or minimize a scheduling objective. Instead, its method is a stochastic, rule-based simulation of human and AI decision-quality scores. This distinction matters when interpreting the results: the scores are illustrative model outputs, not validated measures of crane productivity or operational safety.

The inputs are a human-operator profile, an AI-system profile, and a set of decision scenarios. The simulation produces decision records, summary metrics, and visualizations. Its purpose is to make the partnership logic inspectable and reproducible, not to prescribe real-world authority or safety policies.

### Modeling notes

- Human and AI performance are represented by bounded quality scores between 0 and 1. These scores are simplified proxies, not measured probabilities unless separately calibrated against outcome data.
- Human decisions and scenario variation use pseudorandom numbers. A fixed random seed makes a run reproducible when the notebook is executed from the beginning in the same software environment.
- Trust and authority are represented by explicit rules in the code. They are not learned from operational evidence, and the trust categories are not a validated psychological scale.
- The simulation treats decisions as successive observations and does not model crane locations, travel times, job precedence, interference, or resource constraints.

## 1. Partnership model

### Intuition

For each scenario, the AI produces a recommendation and the human operator makes a decision with access to that recommendation. A rule selects who leads: the AI, the human, or both. The partnership combines the two quality scores according to that authority rule. A simulated outcome then updates performance and trust-related metrics.

### Quality-score calculations

The human's decision quality is generated from a random baseline, experience, decision style, and possible AI influence:

$$
q_H = \operatorname{clip}\!\left[
    (b_H + e_H)s_H + a_H,
    0, 1
\right],
$$

where $b_H$ is a random baseline score, $e_H$ is the experience bonus, $s_H$ is the decision-style factor, and $a_H$ is the influence of the AI recommendation. The function `HumanOperator.make_human_decision` implements this rule.

The AI's unbounded performance estimate is adjusted for scenario complexity and recent learning, then clipped to the unit interval:

$$
q_A = \operatorname{clip}\!\left[
    \frac{r_A}{c} + \ell_A,
    0, 1
\right],
$$

where $r_A$ is the current AI accuracy parameter, $c$ is scenario complexity, and $\ell_A$ is a learning adjustment based on recent simulated outcomes. The function `AISystem.generate_recommendation` implements this calculation. The resulting score is a simulation proxy; the model does not establish that it is a calibrated probability of success.

The final decision score depends on the selected authority:

$$
q_F =
\begin{cases}
0.9q_A + 0.1q_H, & \text{AI lead},\\
0.8q_H + 0.2q_A, & \text{human lead},\\
\dfrac{q_H+q_A}{2} + b_S, & \text{collaborative},\\
\min(q_H,q_A), & \text{consensus},
\end{cases}
\qquad 0 \leq q_F \leq 1.
$$

Here $q_H$ and $q_A$ are the human and AI scores, and $b_S$ is a small agreement bonus in the collaborative case. The corresponding code is in `SymbioticPartnership._ai_lead_decision`, `_human_lead_decision`, `_collaborative_decision`, and `_consensus_decision`.

### Authority and sequential updates

The decision type first selects a default authority. Trust level and urgency can adjust that choice. After each decision, the simulation draws an outcome by multiplying the final score by a random factor and clipping the result to $[0,1]$. The outcome updates partnership metrics, AI performance history, and the human operator's trust in AI.

This is a sequential simulation rather than an optimization algorithm. For $N$ decisions, the main simulation loop processes each decision once, so its computational work grows linearly with $N$, apart from the bounded history calculations used for recent averages.

In [ ]:
import random
import time
import warnings
from collections import defaultdict
from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np

warnings.filterwarnings("ignore")

# Reproducible pseudorandom choices make the illustrative simulation repeatable.
np.random.seed(42)
random.seed(42)

## 2. Representing the human operator and AI system

The classes below separate profile data from behavior. `HumanOperator` evaluates a recommendation, produces a human decision, and updates trust. `AISystem` generates an explainable recommendation and adjusts its accuracy parameter using recent simulated outcomes. The fields and update rules are deliberately explicit so that each modeled assumption can be inspected.

In [ ]:
class DecisionType(Enum):
    """Decision categories used by the partnership authority rules."""

    ROUTINE_OPTIMIZATION = "routine_optimization"
    EMERGENCY_RESPONSE = "emergency_response"
    STRATEGIC_PLANNING = "strategic_planning"
    ETHICAL_JUDGMENT = "ethical_judgment"
    RESOURCE_ALLOCATION = "resource_allocation"
    CONFLICT_RESOLUTION = "conflict_resolution"


class TrustLevel(Enum):
    """Ordinal partnership trust categories used by the simulation."""

    INITIAL = "initial"
    DEVELOPING = "developing"
    ESTABLISHED = "established"
    SYMBIOTIC = "symbiotic"


class DecisionAuthority(Enum):
    """Which participant has the primary decision authority."""

    HUMAN_LEAD = "human_lead"
    AI_LEAD = "ai_lead"
    COLLABORATIVE = "collaborative"
    CONSENSUS = "consensus"


@dataclass
class HumanOperator:
    """Human operator profile and simplified decision behavior."""

    id: str
    name: str
    experience_years: int
    specialization_areas: List[str]
    decision_style: str
    trust_in_ai: float = 0.5
    confidence_level: float = 0.7
    risk_tolerance: float = 0.5
    learning_rate: float = 0.1

    decisions_made: int = 0
    successful_decisions: int = 0
    ai_agreements: int = 0
    ai_disagreements: int = 0

    expertise_weights: Dict[str, float] = field(
        default_factory=lambda: defaultdict(float)
    )

    def __post_init__(self) -> None:
        """Assign a simple experience-based weight to each specialization."""
        for area in self.specialization_areas:
            self.expertise_weights[area] = min(
                1.0, 0.5 + self.experience_years * 0.05
            )

    def evaluate_ai_recommendation(
        self, ai_recommendation: Dict, context: Dict
    ) -> Tuple[float, str]:
        """Return a confidence score and a verbal assessment of a recommendation."""
        decision_type = ai_recommendation.get("type", "routine_optimization")
        expertise_score = self.expertise_weights.get(decision_type, 0.3)
        experience_factor = min(1.0, self.experience_years / 20.0)
        trust_factor = self.trust_in_ai
        recommendation_quality = ai_recommendation.get("confidence", 0.5)

        confidence = (
            expertise_score * 0.3
            + experience_factor * 0.2
            + trust_factor * 0.2
            + recommendation_quality * 0.3
        )

        if confidence > 0.8:
            assessment = "Strong confidence in the AI recommendation"
        elif confidence > 0.6:
            assessment = "Moderate confidence in the AI recommendation"
        elif confidence > 0.4:
            assessment = "Uncertain about the AI recommendation"
        else:
            assessment = "Low confidence in the AI recommendation"

        return confidence, assessment

    def update_trust(self, ai_performance: float, agreement_occurred: bool) -> None:
        """Adjust trust gradually using simulated performance and agreement."""
        if agreement_occurred:
            trust_adjustment = self.learning_rate * (ai_performance - 0.5) * 0.2
        else:
            trust_adjustment = -self.learning_rate * 0.1

        self.trust_in_ai = max(
            0.1, min(1.0, self.trust_in_ai + trust_adjustment)
        )

    def make_human_decision(
        self,
        problem_context: Dict,
        ai_recommendation: Optional[Dict] = None,
    ) -> Dict:
        """Simulate a human quality score, optionally influenced by the AI."""
        base_decision_score = random.gauss(0.7, 0.1)
        experience_bonus = min(0.2, self.experience_years * 0.01)

        if self.decision_style == "conservative":
            style_factor = 0.9
        elif self.decision_style == "aggressive":
            style_factor = 1.1
        else:
            style_factor = 1.0

        ai_influence = 0.0
        if ai_recommendation:
            confidence, _ = self.evaluate_ai_recommendation(
                ai_recommendation, problem_context
            )
            ai_influence = confidence * self.trust_in_ai * 0.3

            recommendation_quality = ai_recommendation[
                "recommendation_quality"
            ]
            if abs(base_decision_score - recommendation_quality) < 0.2:
                self.ai_agreements += 1
            else:
                self.ai_disagreements += 1

        decision_quality = (base_decision_score + experience_bonus) * style_factor
        decision_quality += ai_influence
        decision_quality = max(0.0, min(1.0, decision_quality))
        self.decisions_made += 1

        return {
            "decision_quality": decision_quality,
            "human_confidence": self.confidence_level,
            "ai_influence": ai_influence,
            "decision_authority": "human",
            "reasoning": (
                f"Human decision based on {self.experience_years} years of experience"
            ),
        }


@dataclass
class AISystem:
    """AI profile with a simple recommendation and outcome-update rule."""

    id: str
    model_type: str
    capabilities: List[str]
    explainability_level: float = 0.7
    accuracy_rate: float = 0.85
    processing_speed: float = 0.9
    learning_rate: float = 0.15

    recommendations_made: int = 0
    successful_recommendations: int = 0
    human_acceptances: int = 0
    human_rejections: int = 0

    decision_history: List[Dict] = field(default_factory=list)
    performance_history: List[float] = field(default_factory=list)

    def generate_recommendation(
        self,
        problem_context: Dict,
        human_profile: Optional[HumanOperator] = None,
    ) -> Dict:
        """Generate a bounded recommendation score and explanatory text."""
        complexity_factor = problem_context.get("complexity", 1.0)
        adjusted_performance = self.accuracy_rate / complexity_factor

        if len(self.performance_history) > 10:
            recent_performance = np.mean(self.performance_history[-10:])
            learning_bonus = (recent_performance - self.accuracy_rate) * 0.1
            adjusted_performance += learning_bonus

        recommendation_quality = max(0.0, min(1.0, adjusted_performance))
        confidence = max(0.3, min(1.0, recommendation_quality))
        explanation = self._generate_explanation(problem_context, confidence)

        if human_profile:
            explanation = self._tailor_explanation(explanation, human_profile)

        self.recommendations_made += 1

        return {
            "recommendation_quality": recommendation_quality,
            "confidence": confidence,
            "explanation": explanation,
            "processing_time": 1.0 / self.processing_speed,
            "decision_authority": "ai",
            "reasoning": f"AI recommendation using the {self.model_type} model",
        }

    def _generate_explanation(self, problem_context: Dict, confidence: float) -> str:
        """Describe the model's score without treating it as calibrated probability."""
        explanations = [
            f"The model assigned a confidence score of {confidence:.1%}",
            f"The recommendation has a simulated confidence score of {confidence:.1%}",
            f"The model's recommendation score is {confidence:.1%} under this scenario",
            f"The model reports a confidence score of {confidence:.1%}",
        ]
        return random.choice(explanations)

    def _tailor_explanation(self, explanation: str, human: HumanOperator) -> str:
        """Add context-sensitive wording based on operator experience."""
        if human.experience_years > 10:
            return f"{explanation}. Review the model assumptions and operational context."
        return f"{explanation}. Compare the recommendation with the scenario details."

    def learn_from_outcome(
        self, decision_outcome: float, human_accepted: bool
    ) -> None:
        """Update the simulated accuracy parameter and acceptance counters."""
        self.performance_history.append(decision_outcome)

        if len(self.performance_history) > 5:
            recent_avg = np.mean(self.performance_history[-5:])
            self.accuracy_rate = 0.7 * self.accuracy_rate + 0.3 * recent_avg
            self.accuracy_rate = max(0.0, min(1.0, self.accuracy_rate))

        if human_accepted:
            self.human_acceptances += 1
        else:
            self.human_rejections += 1

        if decision_outcome > 0.7:
            self.successful_recommendations += 1

## 3. Partnership rules and metric updates

`SymbioticPartnership` coordinates a single decision. It determines authority, requests the AI recommendation, obtains the human score, and applies the matching combination rule. It then records enough information to analyze the run.

Agreement is defined consistently as the absolute difference between human and AI quality scores being less than $0.2$. A simulated outcome above $0.7$ is counted as successful for the human and AI counters; this is a threshold-based proxy, not an independently observed operational success. The collaboration score is the fraction of collaborative or consensus decisions among the latest five decisions.

In [ ]:
@dataclass
class SymbioticPartnership:
    """Coordinate simulated decisions and partnership performance tracking."""

    human_operator: HumanOperator
    ai_system: AISystem
    partnership_id: str

    trust_level: TrustLevel = TrustLevel.INITIAL
    collaboration_score: float = 0.5
    decision_history: List[Dict] = field(default_factory=list)
    performance_metrics: Dict[str, List[float]] = field(
        default_factory=lambda: defaultdict(list)
    )

    adaptation_rate: float = 0.1
    synergy_factor: float = 0.0

    def determine_decision_authority(
        self, decision_type: DecisionType, urgency: float
    ) -> DecisionAuthority:
        """Select authority from decision type, partnership trust, and urgency."""
        authority_mapping = {
            DecisionType.ROUTINE_OPTIMIZATION: DecisionAuthority.AI_LEAD,
            DecisionType.EMERGENCY_RESPONSE: DecisionAuthority.COLLABORATIVE,
            DecisionType.STRATEGIC_PLANNING: DecisionAuthority.HUMAN_LEAD,
            DecisionType.ETHICAL_JUDGMENT: DecisionAuthority.HUMAN_LEAD,
            DecisionType.RESOURCE_ALLOCATION: DecisionAuthority.COLLABORATIVE,
            DecisionType.CONFLICT_RESOLUTION: DecisionAuthority.CONSENSUS,
        }

        base_authority = authority_mapping.get(
            decision_type, DecisionAuthority.COLLABORATIVE
        )

        if self.trust_level == TrustLevel.INITIAL:
            if base_authority == DecisionAuthority.AI_LEAD:
                return DecisionAuthority.COLLABORATIVE
        elif self.trust_level == TrustLevel.SYMBIOTIC:
            if (
                base_authority == DecisionAuthority.HUMAN_LEAD
                and decision_type == DecisionType.ROUTINE_OPTIMIZATION
            ):
                return DecisionAuthority.AI_LEAD

        if urgency > 0.8 and base_authority == DecisionAuthority.CONSENSUS:
            return DecisionAuthority.COLLABORATIVE

        return base_authority

    def collaborative_decision_making(self, problem_context: Dict) -> Dict:
        """Run one human–AI decision cycle and store its component scores."""
        decision_type = DecisionType(
            problem_context.get("type", "routine_optimization")
        )
        urgency = problem_context.get("urgency", 0.5)
        authority = self.determine_decision_authority(decision_type, urgency)

        ai_recommendation = self.ai_system.generate_recommendation(
            problem_context, self.human_operator
        )
        human_decision = self.human_operator.make_human_decision(
            problem_context, ai_recommendation
        )

        if authority == DecisionAuthority.AI_LEAD:
            final_decision = self._ai_lead_decision(
                ai_recommendation, human_decision
            )
        elif authority == DecisionAuthority.HUMAN_LEAD:
            final_decision = self._human_lead_decision(
                human_decision, ai_recommendation
            )
        elif authority == DecisionAuthority.COLLABORATIVE:
            final_decision = self._collaborative_decision(
                human_decision, ai_recommendation
            )
        else:
            final_decision = self._consensus_decision(
                human_decision, ai_recommendation
            )

        decision_record = {
            "timestamp": time.time(),
            "decision_type": decision_type.value,
            "authority": authority.value,
            "human_decision": human_decision,
            "ai_recommendation": ai_recommendation,
            "final_decision": final_decision,
            "problem_context": problem_context,
        }
        self.decision_history.append(decision_record)
        return final_decision

    def _ai_lead_decision(self, ai_rec: Dict, human_dec: Dict) -> Dict:
        """Give the AI primary weight while retaining a human contribution."""
        human_validation = human_dec["decision_quality"] * 0.1
        final_quality = ai_rec["recommendation_quality"] * 0.9 + human_validation
        final_quality = max(0.0, min(1.0, final_quality))

        return {
            "decision_quality": final_quality,
            "decision_authority": "ai_lead",
            "human_validation": human_validation,
            "ai_confidence": ai_rec["confidence"],
            "reasoning": f"AI-led decision with human validation: {ai_rec['reasoning']}",
        }

    def _human_lead_decision(self, human_dec: Dict, ai_rec: Dict) -> Dict:
        """Give the human primary weight while retaining AI assistance."""
        ai_assistance = ai_rec["recommendation_quality"] * 0.2
        final_quality = human_dec["decision_quality"] * 0.8 + ai_assistance
        final_quality = max(0.0, min(1.0, final_quality))

        return {
            "decision_quality": final_quality,
            "decision_authority": "human_lead",
            "human_confidence": human_dec["human_confidence"],
            "ai_assistance": ai_assistance,
            "reasoning": f"Human-led decision with AI assistance: {human_dec['reasoning']}",
        }

    def _collaborative_decision(self, human_dec: Dict, ai_rec: Dict) -> Dict:
        """Combine human and AI scores equally and add a small agreement bonus."""
        final_quality = (
            human_dec["decision_quality"] + ai_rec["recommendation_quality"]
        ) / 2

        agreement_bonus = 0.0
        if (
            abs(
                human_dec["decision_quality"]
                - ai_rec["recommendation_quality"]
            )
            < 0.2
        ):
            agreement_bonus = 0.1
            self.synergy_factor = min(1.0, self.synergy_factor + 0.01)

        final_quality = max(0.0, min(1.0, final_quality + agreement_bonus))

        return {
            "decision_quality": final_quality,
            "decision_authority": "collaborative",
            "human_contribution": human_dec["decision_quality"],
            "ai_contribution": ai_rec["recommendation_quality"],
            "agreement_bonus": agreement_bonus,
            "reasoning": "Collaborative decision combining human and AI scores",
        }

    def _consensus_decision(self, human_dec: Dict, ai_rec: Dict) -> Dict:
        """Use the lower component score as the consensus quality score."""
        final_quality = min(
            human_dec["decision_quality"], ai_rec["recommendation_quality"]
        )

        return {
            "decision_quality": final_quality,
            "decision_authority": "consensus",
            "human_approval": human_dec["decision_quality"],
            "ai_approval": ai_rec["recommendation_quality"],
            "consensus_strength": final_quality,
            "reasoning": "Consensus decision requiring human and AI approval",
        }

    def update_partnership_metrics(self, decision_outcome: float) -> None:
        """Update performance, trust, learning, and acceptance metrics."""
        if not self.decision_history:
            raise ValueError("A decision must be recorded before metrics are updated.")

        decision_outcome = max(0.0, min(1.0, decision_outcome))
        self.performance_metrics["decision_quality"].append(decision_outcome)
        self.performance_metrics["collaboration_score"].append(
            self.collaboration_score
        )
        self.performance_metrics["synergy_factor"].append(self.synergy_factor)

        if decision_outcome > 0.7:
            self.human_operator.successful_decisions += 1

        if len(self.decision_history) > 5:
            recent_decisions = self.decision_history[-5:]
            collaborative_count = sum(
                1
                for decision in recent_decisions
                if decision["final_decision"]["decision_authority"]
                in ["collaborative", "consensus"]
            )
            self.collaboration_score = collaborative_count / 5

        if decision_outcome > 0.8:
            self.trust_level = self._advance_trust_level()

        last_decision = self.decision_history[-1]
        human_quality = last_decision["human_decision"]["decision_quality"]
        ai_quality = last_decision["ai_recommendation"]["recommendation_quality"]
        agreement_occurred = abs(human_quality - ai_quality) < 0.2

        self.ai_system.learn_from_outcome(
            decision_outcome, human_accepted=agreement_occurred
        )
        self.human_operator.update_trust(ai_quality, agreement_occurred)
        last_decision["trust_level"] = self.trust_level.value

    def _advance_trust_level(self) -> TrustLevel:
        """Advance the ordinal trust category when collaboration is sustained."""
        if (
            self.trust_level == TrustLevel.INITIAL
            and self.collaboration_score > 0.6
        ):
            return TrustLevel.DEVELOPING
        if (
            self.trust_level == TrustLevel.DEVELOPING
            and self.collaboration_score > 0.7
        ):
            return TrustLevel.ESTABLISHED
        if (
            self.trust_level == TrustLevel.ESTABLISHED
            and self.collaboration_score > 0.8
        ):
            return TrustLevel.SYMBIOTIC
        return self.trust_level

    def get_partnership_analysis(self) -> Dict[str, Any]:
        """Summarize the decisions and metrics recorded in this run."""
        if not self.decision_history:
            return {"status": "No decisions made yet"}

        total_decisions = len(self.decision_history)
        avg_decision_quality = np.mean(
            [
                decision["final_decision"]["decision_quality"]
                for decision in self.decision_history
            ]
        )

        authority_counts = defaultdict(int)
        agreements = 0
        for decision in self.decision_history:
            authority_counts[
                decision["final_decision"]["decision_authority"]
            ] += 1
            human_quality = decision["human_decision"]["decision_quality"]
            ai_quality = decision["ai_recommendation"]["recommendation_quality"]
            if abs(human_quality - ai_quality) < 0.2:
                agreements += 1

        agreement_rate = agreements / total_decisions
        if len(self.performance_metrics["decision_quality"]) > 10:
            recent_performance = np.mean(
                self.performance_metrics["decision_quality"][-5:]
            )
            early_performance = np.mean(
                self.performance_metrics["decision_quality"][:5]
            )
            improvement_trend = recent_performance - early_performance
        else:
            improvement_trend = 0.0

        return {
            "total_decisions": total_decisions,
            "avg_decision_quality": avg_decision_quality,
            "trust_level": self.trust_level.value,
            "collaboration_score": self.collaboration_score,
            "synergy_factor": self.synergy_factor,
            "agreement_rate": agreement_rate,
            "authority_distribution": dict(authority_counts),
            "improvement_trend": improvement_trend,
            "human_trust_in_ai": self.human_operator.trust_in_ai,
            "ai_accuracy": self.ai_system.accuracy_rate,
        }

## 4. Create the example partnership

The profiles and parameters in this cell define the main example. The human operator has specified experience, specializations, and initial trust; the AI has a model type, capabilities, and initial accuracy parameter. These are inputs to the simulation, not empirical estimates of a real terminal team.

In [ ]:
def create_symbiotic_scenario() -> SymbioticPartnership:
    """Create the example human operator, AI system, and partnership."""
    print("=== HUMAN-AI PARTNERSHIP SCENARIO ===")

    human_operator = HumanOperator(
        id="operator_001",
        name="Sarah Chen",
        experience_years=12,
        specialization_areas=[
            "strategic_planning",
            "emergency_response",
            "resource_allocation",
        ],
        decision_style="balanced",
        trust_in_ai=0.6,
        confidence_level=0.8,
        risk_tolerance=0.5,
    )

    ai_system = AISystem(
        id="ai_optimizer_001",
        model_type="hybrid",
        capabilities=[
            "optimization",
            "prediction",
            "pattern_recognition",
            "explainability",
        ],
        explainability_level=0.8,
        accuracy_rate=0.85,
        processing_speed=0.95,
    )

    partnership = SymbioticPartnership(
        human_operator=human_operator,
        ai_system=ai_system,
        partnership_id="partnership_001",
    )

    print("Created partnership:")
    print(
        f"  Human: {human_operator.name} "
        f"({human_operator.experience_years} years of experience)"
    )
    print(f"    Specializations: {', '.join(human_operator.specialization_areas)}")
    print(f"    Decision style: {human_operator.decision_style}")
    print(f"    Initial trust in AI: {human_operator.trust_in_ai:.2f}")
    print(f"  AI: {ai_system.id} ({ai_system.model_type} model)")
    print(f"    Capabilities: {', '.join(ai_system.capabilities)}")
    print(f"    Initial accuracy parameter: {ai_system.accuracy_rate:.2f}")
    print(f"    Explainability setting: {ai_system.explainability_level:.2f}")
    print(f"  Partnership: {partnership.partnership_id}")
    print(f"    Initial trust category: {partnership.trust_level.value}")
    print(f"    Initial collaboration score: {partnership.collaboration_score:.2f}")

    return partnership


symbiotic_partnership = create_symbiotic_scenario()

## 5. Simulate a sequence of decisions

Each iteration selects a scenario, varies its complexity and urgency, runs the partnership decision rules, and draws a simulated outcome. Periodic reports expose the state of the simulation while it runs. The final authority proportions are descriptive summaries of this one seeded run; they are not general performance guarantees.

In [ ]:
def simulate_symbiotic_operations(
    partnership: SymbioticPartnership, num_decisions: int = 50
) -> Dict[str, Any]:
    """Simulate partnership decisions and print periodic summary metrics."""
    if num_decisions <= 0:
        raise ValueError("num_decisions must be a positive integer.")

    print("\n=== SIMULATING PARTNERSHIP OPERATIONS ===")
    print(f"Number of decisions to simulate: {num_decisions}\n")

    decision_scenarios = [
        {
            "type": "routine_optimization",
            "complexity": 1.0,
            "urgency": 0.3,
            "description": "Routine crane scheduling optimization",
        },
        {
            "type": "emergency_response",
            "complexity": 1.5,
            "urgency": 0.9,
            "description": "Emergency equipment failure response",
        },
        {
            "type": "strategic_planning",
            "complexity": 2.0,
            "urgency": 0.2,
            "description": "Long-term capacity planning",
        },
        {
            "type": "ethical_judgment",
            "complexity": 1.8,
            "urgency": 0.7,
            "description": "Ethical resource allocation decision",
        },
        {
            "type": "resource_allocation",
            "complexity": 1.3,
            "urgency": 0.6,
            "description": "Crane resource allocation decision",
        },
        {
            "type": "conflict_resolution",
            "complexity": 1.6,
            "urgency": 0.8,
            "description": "Inter-departmental conflict resolution",
        },
    ]

    for i in range(num_decisions):
        scenario = random.choice(decision_scenarios)
        context = {
            "type": scenario["type"],
            "complexity": scenario["complexity"] * random.uniform(0.8, 1.2),
            "urgency": scenario["urgency"] * random.uniform(0.8, 1.2),
            "description": scenario["description"],
        }

        decision = partnership.collaborative_decision_making(context)
        outcome = decision["decision_quality"] * random.uniform(0.8, 1.2)
        outcome = max(0.0, min(1.0, outcome))
        partnership.update_partnership_metrics(outcome)

        if (i + 1) % 10 == 0:
            print(f"--- Decision {i + 1} ---")
            print(f"Scenario: {context['description']}")
            print(f"Decision authority: {decision['decision_authority']}")
            print(f"Decision quality score: {decision['decision_quality']:.3f}")
            print(f"Simulated outcome score: {outcome:.3f}")
            print(f"Trust category: {partnership.trust_level.value}")
            print(f"Collaboration score: {partnership.collaboration_score:.3f}")
            print(f"Human trust in AI: {partnership.human_operator.trust_in_ai:.3f}")
            print()

    print("=== SIMULATION COMPLETE ===")
    analysis = partnership.get_partnership_analysis()

    print("\nFinal Partnership Analysis:")
    print(f"  Total decisions: {analysis['total_decisions']}")
    print(f"  Average decision quality score: {analysis['avg_decision_quality']:.3f}")
    print(f"  Final trust category: {analysis['trust_level']}")
    print(f"  Final collaboration score: {analysis['collaboration_score']:.3f}")
    print(f"  Final synergy factor: {analysis['synergy_factor']:.3f}")
    print(f"  Human-AI agreement rate: {analysis['agreement_rate']:.3f}")
    print(f"  Improvement trend: {analysis['improvement_trend']:+.3f}")

    print("\nAuthority Distribution:")
    for authority, count in analysis["authority_distribution"].items():
        percentage = count / analysis["total_decisions"] * 100
        print(f"  {authority}: {count} ({percentage:.1f}%)")

    print("\nIndividual Performance Counters:")
    print(f"  Human decisions made: {partnership.human_operator.decisions_made}")
    print(
        "  Human outcomes above the success threshold: "
        f"{partnership.human_operator.successful_decisions}"
    )
    print(f"  Human-AI agreements: {partnership.human_operator.ai_agreements}")
    print(f"  Human-AI disagreements: {partnership.human_operator.ai_disagreements}")
    print(f"  AI recommendations: {partnership.ai_system.recommendations_made}")
    print(
        "  Outcomes above the AI success threshold: "
        f"{partnership.ai_system.successful_recommendations}"
    )
    print(f"  Recommendations classified as accepted: {partnership.ai_system.human_acceptances}")
    print(f"  Recommendations classified as rejected: {partnership.ai_system.human_rejections}")

    return analysis


partnership_analysis = simulate_symbiotic_operations(
    symbiotic_partnership, num_decisions=50
)

## 6. Visualize the simulated results

The panels show decision-quality scores, the recorded ordinal trust category, authority counts, and a rolling agreement rate. The trust plot uses the category recorded after each outcome update; it does not infer trust from decision quality. The rolling agreement rate uses the same $0.2$ score-difference threshold as the partnership metrics.

In [ ]:
def visualize_symbiotic_results(
    analysis: Dict, partnership: SymbioticPartnership
) -> None:
    """Plot performance, recorded trust, authority, and agreement summaries."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(
        "Human-AI Partnership: Simulated Collaboration Metrics",
        fontsize=16,
        fontweight="bold",
    )

    ax1 = axes[0, 0]
    decision_qualities = partnership.performance_metrics["decision_quality"]
    if decision_qualities:
        decision_numbers = range(1, len(decision_qualities) + 1)
        ax1.plot(
            decision_numbers,
            decision_qualities,
            color="blue",
            linewidth=1.5,
            alpha=0.7,
            label="Simulated outcome score",
        )
        if len(decision_qualities) > 5:
            window_size = 5
            moving_average = [
                np.mean(decision_qualities[max(0, i - window_size + 1) : i + 1])
                for i in range(len(decision_qualities))
            ]
            ax1.plot(
                decision_numbers,
                moving_average,
                color="red",
                linewidth=2.5,
                label="Trailing 5-decision average",
            )
        ax1.set_xlabel("Decision number")
        ax1.set_ylabel("Score (0 to 1)")
        ax1.set_title("Simulated Outcome Scores")
        ax1.legend(loc="best")
        ax1.grid(True, alpha=0.3)
        ax1.set_ylim(0, 1)
    else:
        ax1.set_title("No decision scores are available")

    ax2 = axes[0, 1]
    trust_levels = [
        TrustLevel.INITIAL,
        TrustLevel.DEVELOPING,
        TrustLevel.ESTABLISHED,
        TrustLevel.SYMBIOTIC,
    ]
    trust_to_value = {level.value: index for index, level in enumerate(trust_levels)}
    trust_values = [
        trust_to_value[decision["trust_level"]]
        for decision in partnership.decision_history
        if "trust_level" in decision
    ]
    if trust_values:
        ax2.step(
            range(1, len(trust_values) + 1),
            trust_values,
            where="post",
            color="green",
            linewidth=2,
            label="Recorded partnership trust category",
        )
        ax2.set_xlabel("Decision number")
        ax2.set_ylabel("Ordinal trust category")
        ax2.set_title("Recorded Trust-Category Progression")
        ax2.set_yticks(range(len(trust_levels)))
        ax2.set_yticklabels([level.value.title() for level in trust_levels])
        ax2.legend(loc="best")
        ax2.grid(True, alpha=0.3)
    else:
        ax2.set_title("No trust-category records are available")

    ax3 = axes[1, 0]
    authority_distribution = analysis.get("authority_distribution", {})
    if authority_distribution:
        authorities = list(authority_distribution)
        counts = [authority_distribution[authority] for authority in authorities]
        colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
        bars = ax3.bar(
            authorities,
            counts,
            color=colors[: len(authorities)],
            label="Decision count",
        )
        total_decisions = analysis["total_decisions"]
        for bar, count in zip(bars, counts):
            percentage = count / total_decisions * 100
            ax3.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.5,
                f"{percentage:.1f}%",
                ha="center",
                va="bottom",
            )
        ax3.set_xlabel("Decision authority")
        ax3.set_ylabel("Number of decisions")
        ax3.set_title("Decision-Authority Distribution")
        ax3.tick_params(axis="x", rotation=35)
        ax3.legend(loc="best")
        ax3.grid(True, alpha=0.3, axis="y")
    else:
        ax3.set_title("No authority records are available")

    ax4 = axes[1, 1]
    agreements_over_time = []
    window_size = 10
    for i in range(len(partnership.decision_history)):
        start_index = max(0, i - window_size + 1)
        window_decisions = partnership.decision_history[start_index : i + 1]
        agreements = sum(
            1
            for decision in window_decisions
            if abs(
                decision["human_decision"]["decision_quality"]
                - decision["ai_recommendation"]["recommendation_quality"]
            )
            < 0.2
        )
        agreements_over_time.append(agreements / len(window_decisions))

    if agreements_over_time:
        decision_numbers = range(1, len(agreements_over_time) + 1)
        ax4.plot(
            decision_numbers,
            agreements_over_time,
            color="purple",
            linewidth=2,
            label="Trailing agreement rate (up to 10 decisions)",
        )
        ax4.fill_between(
            decision_numbers,
            0,
            agreements_over_time,
            alpha=0.2,
            color="purple",
        )
        ax4.set_xlabel("Decision number")
        ax4.set_ylabel("Agreement rate (0 to 1)")
        ax4.set_title("Human-AI Agreement Over Time")
        ax4.set_ylim(0, 1)
        ax4.legend(loc="best")
        ax4.grid(True, alpha=0.3)
    else:
        ax4.set_title("No agreement records are available")

    plt.tight_layout()
    plt.show()


visualize_symbiotic_results(partnership_analysis, symbiotic_partnership)

## 7. Evaluate partnership effectiveness

This analysis compares the human score, AI recommendation score, and combined score recorded for each simulated decision. The quantity called the **synergy effect** is the difference between the observed combined average and the arithmetic mean of the two component averages. It is a descriptive comparison within this simulation, not proof that collaboration causes better outcomes.

The reported ROI is also a score-based comparison, not a financial return: the model contains no costs, revenue, or counterfactual experiment. Decision-type averages may vary because each type is sampled a different number of times.

In [ ]:
def analyze_symbiotic_effectiveness(
    partnership: SymbioticPartnership,
) -> Dict[str, Any]:
    """Compare component and combined scores and summarize partnership metrics."""
    if not partnership.decision_history:
        raise ValueError("At least one decision is required for effectiveness analysis.")

    print("\n=== PARTNERSHIP EFFECTIVENESS ANALYSIS ===")

    human_performance = []
    ai_performance = []
    collaborative_performance = []

    for decision in partnership.decision_history:
        human_performance.append(
            decision["human_decision"]["decision_quality"]
        )
        ai_performance.append(
            decision["ai_recommendation"]["recommendation_quality"]
        )
        collaborative_performance.append(
            decision["final_decision"]["decision_quality"]
        )

    avg_human = np.mean(human_performance)
    avg_ai = np.mean(ai_performance)
    avg_collaborative = np.mean(collaborative_performance)
    expected_collaborative = (avg_human + avg_ai) / 2
    synergy_effect = avg_collaborative - expected_collaborative
    synergy_percentage = (
        synergy_effect / expected_collaborative * 100
        if expected_collaborative > 0
        else 0.0
    )

    print("\nPerformance Score Comparison:")
    print(f"  Human decision average: {avg_human:.3f}")
    print(f"  AI recommendation average: {avg_ai:.3f}")
    print(f"  Combined decision average: {avg_collaborative:.3f}")
    print(f"  Arithmetic-mean reference: {expected_collaborative:.3f}")
    print(f"  Descriptive synergy effect: {synergy_effect:+.3f} ({synergy_percentage:+.1f}%)")

    print("\nAverage Combined Score by Decision Type:")
    type_performance = defaultdict(list)
    for decision in partnership.decision_history:
        type_performance[decision["decision_type"]].append(
            decision["final_decision"]["decision_quality"]
        )

    for decision_type, performances in type_performance.items():
        print(
            f"  {decision_type}: {np.mean(performances):.3f} "
            f"({len(performances)} decisions)"
        )

    print("\nTrust Development:")
    print("  Initial human trust in AI: 0.600")
    print(
        f"  Final human trust in AI: "
        f"{partnership.human_operator.trust_in_ai:.3f}"
    )
    print(
        f"  Change in human trust: "
        f"{partnership.human_operator.trust_in_ai - 0.6:+.3f}"
    )
    print(f"  Final partnership trust category: {partnership.trust_level.value}")

    print("\nSimulated Learning Metrics:")
    print("  Initial AI accuracy parameter: 0.850")
    print(f"  Final AI accuracy parameter: {partnership.ai_system.accuracy_rate:.3f}")
    print(
        f"  Change in AI accuracy parameter: "
        f"{partnership.ai_system.accuracy_rate - 0.85:+.3f}"
    )
    agreement_rate = partnership.get_partnership_analysis()["agreement_rate"]
    print(f"  Human-AI agreement rate: {agreement_rate:.3f}")
    print(f"  Synergy factor: {partnership.synergy_factor:.3f}")

    baseline_performance = max(avg_human, avg_ai)
    improvement_over_baseline = avg_collaborative - baseline_performance
    roi_percentage = (
        improvement_over_baseline / baseline_performance * 100
        if baseline_performance > 0
        else 0.0
    )

    print("\nScore-Based Comparison with the Stronger Component:")
    print(f"  Stronger individual average: {baseline_performance:.3f}")
    print(f"  Combined decision average: {avg_collaborative:.3f}")
    print(f"  Difference from stronger component: {improvement_over_baseline:+.3f}")
    print(f"  Relative score difference: {roi_percentage:+.1f}%")
    print("  This relative score difference is not a financial ROI.")

    maturity_score = (
        partnership.collaboration_score
        + partnership.human_operator.trust_in_ai
        + partnership.synergy_factor
    ) / 3

    if maturity_score > 0.8:
        maturity_level = "Symbiotic Excellence"
    elif maturity_score > 0.7:
        maturity_level = "Highly Effective"
    elif maturity_score > 0.6:
        maturity_level = "Developing"
    else:
        maturity_level = "Initial"

    print("\nRule-Based Partnership Maturity Summary:")
    print(f"  Maturity score: {maturity_score:.3f}")
    print(f"  Maturity category: {maturity_level}")

    return {
        "avg_human_performance": avg_human,
        "avg_ai_performance": avg_ai,
        "avg_collaborative_performance": avg_collaborative,
        "synergy_effect": synergy_effect,
        "synergy_percentage": synergy_percentage,
        "roi_percentage": roi_percentage,
        "maturity_score": maturity_score,
        "maturity_level": maturity_level,
    }


effectiveness_analysis = analyze_symbiotic_effectiveness(symbiotic_partnership)

## 8. Interpretation and limitations

A human–AI partnership can combine human contextual judgment with computational support, but the quality of that partnership depends on the decision rules, the reliability of each participant's information, and the consequences of errors. In this simulation, authority is assigned by explicit rules, while trust and performance are updated using simplified score-based rules.

The model can help readers trace how those rules interact. It cannot establish that human–AI collaboration is superior to either participant acting alone, because the scores are generated by the same simplified simulation and there is no independently validated benchmark. Likewise, the maturity category is a rule-based summary rather than a standardized measure.

### Potential extensions

A natural extension would be to connect this partnership layer to a separately specified crane-scheduling model, using scheduling outcomes as evidence for decision quality. Additional operational constraints, calibrated performance data, and safety procedures could then be represented explicitly. Those extensions would require data and modeling assumptions beyond this illustrative simulation.